## Stratified 5-fold CV split

Generates the stratified 5-fold partitions used for cross-validation in the thesis. Each fold ends up with a 60/20/20 train/val/test ratio with class proportions preserved across all three subsets.

Two independent splits are generated from the same authentic dataset using different random seeds: one for hyperparameter tuning (seed 42) and one for the final evaluation reported in the thesis (seed 123). Toggle `SPLIT_MODE` to switch between them.

The script copies files into a `fold_{i}/{train|val|test}/{class}/` directory structure that can be read directly by Keras'  standard image-folder loaders.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split

SPLIT_MODE = "eval"  # "eval" or "tuning"

SEED = {"tuning": 42, "eval": 123}[SPLIT_MODE]
N_SPLITS = 5
VAL_FRACTION = 0.25  # 25% of the trainval 80% = 20% of total, giving 60/20/20

class_names = ["t72nolabel", "t80nolabel", "t90nolabel"]
SOURCE_DIR = Path(r"F:\Users\nikol\Desktop\Data\Pictures\RealData")
OUTPUT_DIR = Path(rf"F:\Users\nikol\Desktop\Data\Pictures\authentic_split_cv_{SPLIT_MODE}")


def create_5fold_cv_split(source_dir, output_dir, n_splits=5, val_fraction=0.25, seed=SEED):
    """Build a stratified 5-fold CV split with a 60/20/20 train/val/test ratio per fold."""
    if output_dir.exists():
        print(f"Removing existing output directory: {output_dir}")
        shutil.rmtree(output_dir)

    all_files, all_labels = [], []
    for class_idx, cls in enumerate(class_names):
        files = sorted(
            f for f in (source_dir / cls).iterdir()
            if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")
        )
        all_files.extend(files)
        all_labels.extend([class_idx] * len(files))

    all_files = np.array(all_files)
    all_labels = np.array(all_labels)

    print(f"Total images: {len(all_files)}")
    for class_idx, cls in enumerate(class_names):
        print(f"  {cls}: {np.sum(all_labels == class_idx)}")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    summary = {}

    for fold_idx, (trainval_indices, test_indices) in enumerate(skf.split(all_files, all_labels)):
        fold_dir = output_dir / f"fold_{fold_idx}"
        for split in ["train", "val", "test"]:
            for cls in class_names:
                (fold_dir / split / cls).mkdir(parents=True, exist_ok=True)

        trainval_files = all_files[trainval_indices]
        trainval_labels = all_labels[trainval_indices]

        # seed + fold_idx so train/val partitions differ across folds.
        train_indices, val_indices = train_test_split(
            np.arange(len(trainval_files)),
            test_size=val_fraction,
            stratify=trainval_labels,
            random_state=seed + fold_idx,
        )

        train_files = trainval_files[train_indices]
        train_labels = trainval_labels[train_indices]
        val_files = trainval_files[val_indices]
        val_labels = trainval_labels[val_indices]
        test_files = all_files[test_indices]
        test_labels = all_labels[test_indices]

        for f, lbl in zip(train_files, train_labels):
            shutil.copy2(f, fold_dir / "train" / class_names[lbl] / f.name)
        for f, lbl in zip(val_files, val_labels):
            shutil.copy2(f, fold_dir / "val" / class_names[lbl] / f.name)
        for f, lbl in zip(test_files, test_labels):
            shutil.copy2(f, fold_dir / "test" / class_names[lbl] / f.name)

        fold_summary = {
            "train": {cls: int(np.sum(train_labels == i)) for i, cls in enumerate(class_names)},
            "val": {cls: int(np.sum(val_labels == i)) for i, cls in enumerate(class_names)},
            "test": {cls: int(np.sum(test_labels == i)) for i, cls in enumerate(class_names)},
        }
        summary[f"fold_{fold_idx}"] = fold_summary

        print(f"\nFold {fold_idx}:")
        for split in ["train", "val", "test"]:
            total = sum(fold_summary[split].values())
            pct = total / len(all_files) * 100
            counts = ", ".join(f"{cls}={fold_summary[split][cls]}" for cls in class_names)
            print(f"  {split} (total {total}, {pct:.1f}%): {counts}")

    return summary


print(f"Starting 5-fold CV split ({SPLIT_MODE})")
print(f"Source: {SOURCE_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Seed:   {SEED}\n")

cv_summary = create_5fold_cv_split(
    SOURCE_DIR,
    OUTPUT_DIR,
    n_splits=N_SPLITS,
    val_fraction=VAL_FRACTION,
    seed=SEED,
)

print("\nDone.")